In [1]:
!pip install entsoe-py requests pandas
!pip install matplotlib seaborn scikit-learn
import pandas as pd
import requests
import time
from entsoe import EntsoePandasClient


import pandas as pd
import requests
import time
from entsoe import EntsoePandasClient


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\jarll\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Users\jarll\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
# import pandas as pd
# import requests

# # Hämta t.ex. ett helt år (2024) från Nord Pool-spegeln
# # Format: https://www.elprisetjustnu.se/api/v1/prices/{år}/{månad}-{dag}_SE3.json
# dates = pd.date_range("2024-01-01", "2024-01-05")  # Ändra spann efter behov
# all_prices = []

# for date in dates:
#     url = f"https://www.elprisetjustnu.se/api/v1/prices/{date.strftime('%Y/%m-%d')}_SE3.json"
#     r = requests.get(url)
#     if r.status_code == 200:
#         all_prices.extend(r.json())

# df_prices = pd.DataFrame(all_prices)
# print(df_prices)
# df_prices = df_prices[["time_start", "EUR_per_kWh"]].rename(
#     columns={"time_start": "timestamp", "EUR_per_kWh": "spot_price_eur_kwh"}
# )
# df_prices["spot_price_eur_mwh"] = df_prices["spot_price_eur_kwh"] * 1000
# df_prices["timestamp"] = pd.to_datetime(df_prices["timestamp"])

# #print(df_prices.head())



In [ ]:


# ==========================================
# 1. KONFIGURATION
# ==========================================
API_KEY = "1d03253c-7e54-40ec-9a1b-eda79c13d1b3".strip()
AREA_CODE = "SE_3" # Vi kanske ska göra flera dataset för att hantera flera zoner , just nu har vi bara stockholms området 

#START_DATE = "2022-01-01"
START_DATE = "2025-01-01"
END_DATE = "2026-01-01"  # Justera efter hur långt fram ni vill hämta

LAT = 59.3293  # Stockholm
LON = 18.0686

# ==========================================
# 2. HÄMTA ELPRISER (ENTSO-E i 6-månadersblock)
# ==========================================
print("Hämtar elpriser från ENTSO-E...")
client = EntsoePandasClient(api_key="ac746b47-0403-480a-b87d-41f1ec766a1c")
price_frames = []

# Dela upp i halvår för att undvika 400 Client Error / timeout
date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="3MS", tz="UTC")

for i in range(len(date_range) - 1):
    start_ts = date_range[i]
    end_ts = date_range[i + 1]
    
    try:
        series = client.query_day_ahead_prices(AREA_CODE, start=start_ts, end=end_ts)
        df_block = series.reset_index()
        df_block.columns = ["timestamp", "spot_price_eur_mwh"]
        df_block["timestamp"] = pd.to_datetime(df_block["timestamp"]).dt.tz_convert("UTC")
        price_frames.append(df_block)
        print(f"  -> Klart: {start_ts.strftime('%Y-%m')} till {end_ts.strftime('%Y-%m')} ({len(df_block)} rader)")
    except Exception as e:
        print(f"  -> Fel vid {start_ts.strftime('%Y-%m')}: {e}")
    
    time.sleep(0.3)

df_prices = pd.concat(price_frames, ignore_index=True).drop_duplicates(subset=["timestamp"])

# ==========================================
# 3. HÄMTA VÄDERDATA (Open-Meteo direkt i UTC)
# ==========================================
print("\nHämtar väderdata från Open-Meteo...")
weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": LAT,
    "longitude": LON,
    "start_date": START_DATE,
    "end_date": "2025-12-31",  # Arkiv-API:et har historisk data
    "hourly": "temperature_2m,wind_speed_10m,rain",
    "timezone": "UTC"          # Vi begär direkt i UTC för att undvika sommartidsluckor
}

res = requests.get(weather_url, params=weather_params).json()

if "hourly" not in res:
    raise ValueError(f"Fel från Open-Meteo: {res}")

df_weather = pd.DataFrame(res["hourly"])
df_weather.rename(columns={
    "time": "timestamp",
    "temperature_2m": "temperature_c",
    "wind_speed_10m": "wind_speed_kmh",
    "rain": "rain_mm"
}, inplace=True)

# Sätt tidszon till UTC
df_weather["timestamp"] = pd.to_datetime(df_weather["timestamp"]).dt.tz_localize("UTC")

# ==========================================
# 4. SLÅ IHOP OCH SPARA
# ==========================================
print("\nSlår ihop dataseten...")
df_final = pd.merge(df_weather, df_prices, on="timestamp", how="inner")

# Om ni vill konvertera till svensk lokal tid i slutresultatet:
df_final["timestamp_local"] = df_final["timestamp"].dt.tz_convert("Europe/Stockholm")

print(f"\nKlart! Totalt {len(df_final)} rader genererade.")
print(df_final.head())

df_final.to_csv("se3_weather_and_prices.csv", index=False)
print("Fil sparad: se3_weather_and_prices.csv")

Hämtar elpriser från ENTSO-E...
  -> Klart: 2025-01 till 2025-07 (4345 rader)
  -> Fel vid 2025-07: 599 Server Error: status code 599 for url: https://web-api.tp.entsoe.eu/api?documentType=A44&in_Domain=10Y1001A1001A46L&out_Domain=10Y1001A1001A46L&offset=200&contract_MarketAgreement.type=A01&securityToken=ac746b47-0403-480a-b87d-41f1ec766a1c&periodStart=202506300000&periodEnd=202601020000

Hämtar väderdata från Open-Meteo...

Slår ihop dataseten...

Klart! Totalt 4345 rader genererade.
                  timestamp  temperature_c  wind_speed_kmh  rain_mm  \
0 2025-01-01 00:00:00+00:00           -0.6            18.4      0.0   
1 2025-01-01 01:00:00+00:00            0.2            12.7      0.0   
2 2025-01-01 02:00:00+00:00            0.6             5.2      0.1   
3 2025-01-01 03:00:00+00:00            1.2             6.5      0.1   
4 2025-01-01 04:00:00+00:00            1.5             7.6      0.1   

   spot_price_eur_mwh           timestamp_local  
0                2.47 2025-01-01

In [ ]:
import pandas as pd

# Läs in den sparade datan (tar bråkdelen av en sekund)
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 1. Kolla saknade värden
print("Saknade värden per kolumn:")
print(df.isna().sum())

# 2. Snabb överblick av min/max och medel
print("\nStatistik:")
print(df[["temperature_c", "wind_speed_kmh", "rain_mm", "spot_price_eur_mwh"]].describe().round(2))

# 3. Snabb korrelationsmatris (se hur mycket vinden pressar priset!)
print("\nKorrelation mot elpris:")
print(df[["temperature_c", "wind_speed_kmh", "rain_mm", "spot_price_eur_mwh"]].corr()["spot_price_eur_mwh"].round(3))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 1. Läs in datan
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 2. Rensa eventuella extrema spikar (t.ex. energikris-toppar > 500 EUR) 
# så att regressionslinjen inte förvrängs helt av enstaka extremvärden
clean_df = df[(df["spot_price_eur_mwh"] >= 0) & (df["spot_price_eur_mwh"] <= 300)].dropna(
    subset=["wind_speed_kmh", "spot_price_eur_mwh"]
)

X = clean_df[["wind_speed_kmh"]]
y = clean_df["spot_price_eur_mwh"]

# 3. Träna linjär regression
model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

slope = model.coef_[0]
intercept = model.intercept_
r2 = r2_score(y, y_pred)

print(f"Lutning (koefficient): {slope:.3f} EUR/MWh per km/h vind")
print(f"Intercept: {intercept:.2f} EUR/MWh")
print(f"Förklaringsgrad (R²): {r2:.4f}")

# 4. Skapa visualisering
plt.figure(figsize=(10, 6), dpi=100)

# Eftersom 35 000 punkter blir en gröt kör vi låg alpha (transparens)
sns.regplot(
    data=clean_df.sample(min(5000, len(clean_df))), # Sampla t.ex. 5000 punkter för snabbare och snyggare rendering
    x="wind_speed_kmh",
    y="spot_price_eur_mwh",
    scatter_kws={"alpha": 0.15, "color": "#1f77b4", "s": 15},
    line_kws={"color": "red", "linewidth": 2, "label": f"Trend: y = {slope:.2f}x + {intercept:.1f} (R² = {r2:.3f})"}
)

plt.title("SE3 Elpris vs. Vindhastighet (Linjär Regression)", fontsize=14, fontweight="bold")
plt.xlabel("Vindhastighet vid 10m (km/h)", fontsize=12)
plt.ylabel("Spotpris (EUR/MWh)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="upper right", frameon=True)
plt.tight_layout()

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Läs in data och extrahera tidsfunktioner
df = pd.read_csv("se3_weather_and_prices.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Säkerställ svensk lokal tid så att morgontoppen hamnar runt kl 07–09 och inte förskjuts av UTC
if df["timestamp"].dt.tz is None:
    df["timestamp"] = df["timestamp"].dt.tz_localize("UTC").dt.tz_convert("Europe/Stockholm")
else:
    df["timestamp"] = df["timestamp"].dt.tz_convert("Europe/Stockholm")

df["hour"] = df["timestamp"].dt.hour
df["day_name"] = df["timestamp"].dt.day_name()
df["day_of_week"] = df["timestamp"].dt.dayofweek  # 0 = Måndag, 6 = Söndag
df["is_weekend"] = df["day_of_week"].isin([5, 6]).map({True: "Helg", False: "Vardag"})

# Sortera dagarna i rätt ordning
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
day_labels_sv = ["Mån", "Tis", "Ons", "Tor", "Fre", "Lör", "Sön"]

# 2. Skapa figuren med två delgrafer
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={"width_ratios": [1.2, 1]})

# --- GRAF 1: Heatmap (Veckodag vs Timme) ---
pivot_table = df.pivot_table(
    index="day_name", 
    columns="hour", 
    values="spot_price_eur_mwh", 
    aggfunc="mean"
).reindex(day_order)

sns.heatmap(
    pivot_table, 
    cmap="YlOrRd", 
    cbar_kws={"label": "Medelpris (EUR/MWh)"}, 
    ax=ax1,
    annot=False
)
ax1.set_title("Snittpris: Veckodag vs Timme (SE3)", fontsize=13, fontweight="bold")
ax1.set_xlabel("Timme på dygnet (00–23)", fontsize=11)
ax1.set_ylabel("", fontsize=11)
ax1.set_yticklabels(day_labels_sv, rotation=0)

# --- GRAF 2: Dygnsprofil (Vardag vs Helg) ---
sns.lineplot(
    data=df, 
    x="hour", 
    y="spot_price_eur_mwh", 
    hue="is_weekend", 
    palette={"Vardag": "#d62728", "Helg": "#1f77b4"},
    linewidth=2.5,
    ax=ax2
)
ax2.set_title("Dygnsrytm: Vardag vs Helg", fontsize=13, fontweight="bold")
ax2.set_xlabel("Timme på dygnet (00–23)", fontsize=11)
ax2.set_ylabel("Spotpris (EUR/MWh)", fontsize=11)
ax2.set_xticks(range(0, 24, 2))
ax2.grid(True, linestyle="--", alpha=0.5)
ax2.legend(title="", frameon=True)

plt.tight_layout()
plt.show()